# ELUATE-only — Google Colab

Use this after OCCLUDE has produced the blurred video (music still intact). This notebook does the music-removal pass on its own:

1. Install `eluate` and verify CUDA.
2. Take the OCCLUDE output from `MyDrive/occlude/outputs/`, strip the background music (speech + sfx kept, video stream copied through untouched), and write the final file next to it.

**Runtime (Pro+)**: Runtime → Change runtime type → **A100** (fall back to **L4**). Enable **background execution** so the job survives a closed tab.

**Speed**: ELUATE separates the *audio* track with Bandit v2 and copies the video stream through without re-encoding, so runtime scales with the audio length, not the frame count — this pass is far quicker than the blur pass.

This notebook does **not** assert an exact file size. It only checks the file exists, and cell 6 lists the outputs folder so you can confirm the exact name.

In [ ]:
# 1. Verify GPU (expect A100 / L4 on Pro+)
!nvidia-smi

In [ ]:
# 2. System deps: ffmpeg (audio decode + remux back into the video).
!apt-get -qq install -y ffmpeg > /dev/null

In [ ]:
# 3. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Persist the Bandit v2 checkpoint to Drive so later sessions skip
#    the download. ELUATE's app dir is hardcoded to ~/.eluate; symlink
#    it to Drive.
import os
from pathlib import Path

DRIVE = "/content/drive/MyDrive/occlude"
os.makedirs(f"{DRIVE}/models", exist_ok=True)
os.makedirs(f"{DRIVE}/outputs", exist_ok=True)

home_eluate  = Path.home() / ".eluate"
drive_eluate = Path(f"{DRIVE}/models/eluate")
drive_eluate.mkdir(parents=True, exist_ok=True)
if not home_eluate.exists():
    os.symlink(drive_eluate, home_eluate)

print("eluate cache wired to", drive_eluate)

In [ ]:
# 5. Install eluate, verify CUDA. ELUATE runs Bandit v2 on torch with
#    --device cuda; no onnxruntime repair is needed (that issue is
#    OCCLUDE-specific — InsightFace under ONNX Runtime).
import subprocess, torch

assert torch.cuda.is_available(), "no CUDA - Runtime > Change runtime type > GPU"
print("GPU OK:", torch.cuda.get_device_name(0), flush=True)

subprocess.run("pip install -q -U eluate", shell=True, check=True)

# Confirm the CLI is importable in a FRESH process - that is what the
# eluate subprocess in cell 7 sees.
chk = subprocess.run(
    ["python", "-c",
     "import torch, eluate; "
     "print('eluate', getattr(eluate, '__version__', '?')); "
     "assert torch.cuda.is_available(); print('CUDA OK')"],
    capture_output=True, text=True,
)
print(chk.stdout.strip(), flush=True)
if chk.returncode != 0:
    print(chk.stderr.strip(), flush=True)
    raise SystemExit("eluate import / CUDA check failed - do not start the run")

In [ ]:
# 6. List what's actually in the Drive outputs/ folder, with sizes, so
#    you can copy the EXACT OCCLUDE-output filename into cell 7.
import os
DRIVE = "/content/drive/MyDrive/occlude"
out_dir = f"{DRIVE}/outputs"
assert os.path.isdir(out_dir), f"not found: {out_dir} (is Drive mounted? cell 3)"
for name in sorted(os.listdir(out_dir)):
    p = os.path.join(out_dir, name)
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:8.1f} MB  {name}")

In [ ]:
# 7. RUN. ELUATE on the OCCLUDE output (blurred, music still intact).
#    Strips background music; keeps speech + sfx; copies the video
#    stream through untouched.
import os, shutil, time

DRIVE = "/content/drive/MyDrive/occlude"

# ---- edit these two if the name differs (see cell 6 listing) ----
INPUT_NAME  = "The-Thinking-Game-occluded.mp4"
OUTPUT_NAME = "The-Thinking-Game-final.mp4"
# -----------------------------------------------------------------

IN  = f"{DRIVE}/outputs/{INPUT_NAME}"
OUT = f"{DRIVE}/outputs/{OUTPUT_NAME}"
assert os.path.exists(IN), (
    f"not found: {IN}\n"
    f"run cell 6 and copy the exact name into INPUT_NAME above."
)
print(f"input: {IN}  ({os.path.getsize(IN)/1e6:.1f} MB)", flush=True)

# Process from local disk, not the Drive FUSE mount: heavy reads/writes
# over Drive are slow and flaky on long jobs.
shutil.copy(IN, "/content/elu_in.mp4")

print(">>> eluate (music removal) - runtime scales with audio length", flush=True)
t0 = time.time()
!eluate /content/elu_in.mp4 -o /content/elu_out.mp4 --device cuda
print(f">>> finished in {(time.time()-t0)/60:.1f} min", flush=True)

assert os.path.exists("/content/elu_out.mp4"), "eluate did not produce an output - see log above"
shutil.copy("/content/elu_out.mp4", OUT)
print("DONE:", os.path.getsize(OUT), "bytes ->", OUT, flush=True)